# RFM Customer Segmentation Analysis

In [3]:
from datetime import datetime

# Analysis reference date（Calculate how many days ago this day was, considering it as the present day）
BASE_DATE = datetime(2026, 6, 30)

# Customer purchase log（dummy data）
# Inculud invalid data such as missing values (None) and negative amounts.
rfm_logs = [
    {"customer_id": "C001", "date": "2026-06-25", "amount": 5000},
    {"customer_id": "C002", "date": "2026-05-10", "amount": 12000},
    {"customer_id": "C001", "date": "2026-06-28", "amount": 4000},
    {"customer_id": "C003", "date": "2025-12-01", "amount": 30000},
    {"customer_id": "C004", "date": "2026-06-15", "amount": 15000},
    {"customer_id": "C002", "date": "2026-06-20", "amount": -2000}, # invalid data
    {"customer_id": "C005", "date": "2026-04-01", "amount": 8000},
    {"customer_id": "C001", "date": None,         "amount": 5000},  # invalid data
    {"customer_id": "C005", "date": "2026-05-20", "amount": 12000},
    {"customer_id": "C006", "date": "2026-06-29", "amount": 60000},
]

def is_valid(logs):
    """
    Filters out records with a missing date (None) or a non-positive amount (<= 0),
    and converts the date string to a datetime object.

    Args:
        logs (list): Customer purchase logs. Each log is a dictionary containing customer_id,
        date, and amount.

    Returns:
        list: A list of dictionaries containing customer_id, date(datetime object) and amount.
    """
    valid_logs= []
    for log in logs:
        if log["amount"] > 0 and log["date"] is not None:
            valid_logs.append({
                "customer_id": log["customer_id"],
                "date": datetime.strptime(log["date"], "%Y-%m-%d"),
                "amount": log["amount"]
            })

    return valid_logs

def aggregate_rfm(logs):
    """
    Aggregate RFN (R: Recency, F: Frequency, M: Monetary) data.
    
    Args:
        logs (list): Customer purchase logs. Each log is a dictionary containing customer_id,
        date, and amount.
        
    Returns:
        list: A list of dictionaries containing customer_id, recency, frequency, and monetary.
    """
    valid_logs = is_valid(logs)
    customers = {}
    for log in valid_logs:
        cid = log["customer_id"]
        if cid in customers:
            customers[cid]["monetary"] += log["amount"]
            customers[cid]["frequency"] += 1
            if customers[cid]["date"] < log["date"]:
                customers[cid]["date"] = log["date"]
                customers[cid]["recency"] = (BASE_DATE - customers[cid]["date"]).days

        else:
            customers[cid] = {
                "date": log["date"],
                "recency": (BASE_DATE - log["date"]).days,
                "frequency": 1,
                "monetary": log["amount"]
            }
    
    return [
        {
            "customer_id": cid,
            "recency": data["recency"],
            "frequency": data["frequency"],
            "monetary": data["monetary"]
        }
        for cid, data in customers.items()
    ]

            
